<center><p float="center">
  <img src="https://upload.wikimedia.org/wikipedia/commons/e/e9/4_RGB_McCombs_School_Brand_Branded.png" width="300" height="100"/>
  <img src="https://mma.prnewswire.com/media/1458111/Great_Learning_Logo.jpg?p=facebook" width="200" height="100"/>
</p></center>

<center><font size=10>AI Agents for Business Applications</center></font>
<center><font size=6>Prompt Engineering and Retrieval Augmented Generation - Week 1</center></font>


<center><p float="center">
  <img src="https://i.ibb.co/Q325rK84/medical.png" width="480"/>
</p></center>

<center><font size=6>LLM-Powered Medical Assistant</center></font>


## Problem Statement


### Business Context


The healthcare industry is rapidly evolving, with professionals facing increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. The need for quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.

Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.

To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making, and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.


**Common Questions to Answer**

1. **Critical Care Protocols:** "What is the protocol for managing sepsis in a critical care unit?"

2. **General Surgery:** "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"

3. **Dermatology:** "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"

4. **Neurology:** "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"


### Objective


As an AI specialist, your task is to develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to **understand** issues like information overload, **apply** AI techniques to streamline decision-making, **analyze** its impact on diagnostics and patient outcomes, **evaluate** its potential to standardize care practices, and **create** a functional prototype demonstrating its feasibility and effectiveness.


### Data Description


The **Merck Manuals** are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.


## Installing and Importing Necessary Libraries and Dependencies


In [1]:
# # Install required libraries
# !pip install -q langchain_community==0.3.27 \
#               langchain==0.3.27 \
#               chromadb==1.0.15 \
#               pymupdf==1.26.3 \
#               tiktoken==0.9.0 \
#               datasets==4.0.0 \
#               evaluate==0.4.5 \
#               langchain_openai==0.3.30

**Note**:

- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in **_this notebook_**.


In [2]:
# prompt
# personality: you are medical assistant, you are helpful and precise. You answer the question based on the information provided in the document. If you don't know the answer, say you don't know.
# task definion: You are a medical assistant. You will be given a document and a question. Your task is to answer the question based on the information provided in the document. If you don't know the answer, say you don't know.
# instrunction sometimes step by step:
# 1. Read the document carefully and understand the information provided.
# 2. Read the question carefully and understand what is being asked.
# 3. Look for the relevant information in the document that can help answer the question.
# 4. If you find the relevant information, use it to answer the question.

# chain of thought:
# 1. Read the document carefully and understand the information provided.
# 2. Read the question carefully and understand what is being asked.
# 3. Look for the relevant information in the document that can help answer the question.
# 4. If you find the relevant information, use it to answer the question.

# examples: FEW SHOT EXAMPLE

# output format:
# {
#     "answer": "The answer to the question based on the information provided in the document."
# }

# CONTEXT:
# YOUR DOCUMENT

In [3]:
# Import core libraries
import os  # Interact with the operating system (e.g., set environment variables)
import json  # Read/write JSON data

# Import libraries for working with PDFs and OpenAI
from langchain.document_loaders import (
    PyMuPDFLoader,
)  # Load and extract text from PDF files
from openai import OpenAI  # Access OpenAI's models and services

# Import libraries for processing dataframes and text
import tiktoken  # Tokenizer used for counting and splitting text for models
import pandas as pd  # Load, manipulate, and analyze tabular data

# Import LangChain components for data loading, chunking, embedding, and vector DBs
from langchain.text_splitter import (
    RecursiveCharacterTextSplitter,
)  # Break text into overlapping chunks for processing
from langchain.embeddings.openai import (
    OpenAIEmbeddings,
)  # Create vector embeddings using OpenAI's models  # type: ignore
from langchain.vectorstores import (
    Chroma,
)  # Store and search vector embeddings using Chroma DB  # type: ignore

# from datasets import Dataset                                                    # Used to structure the input (questions, answers, contexts etc.) in tabular format
from langchain_openai import (
    ChatOpenAI,
)  # This is needed since LLM is used in metric computation

/home/alireza/greatlearning/ai_agent_for_business_ut_austin/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Question Answering using LLM


### OpenAI API Calling


In [4]:
# Load the JSON file and extract values
file_name = "../../../config.json"  # Name of the configuration file
with open(file_name, "r") as file:  # Open the config file in read mode
    config = json.load(file)  # Load the JSON content as a dictionary
    API_KEY = config.get("OPENAI_API_KEY")  # Extract the API key from the config
    OPENAI_API_BASE = config.get(
        "OPENAI_API_BASE"
    )  # Extract the OpenAI base URL from the config

# Store API credentials in environment variables
os.environ["OPENAI_API_KEY"] = API_KEY  # Set API key as environment variable
os.environ["OPENAI_BASE_URL"] = (
    OPENAI_API_BASE  # Set API base URL as environment variable
)

# Initialize OpenAI client
client = OpenAI()  # Create an instance of the OpenAI client

### Defining the function to Generate a Response From the LLM


In [5]:
# Define a function to get a response
def response(user_prompt, max_tokens=1000, temperature=0.75, top_p=0.95):
    # Create a chat completion using the OpenAI client
    completion = client.chat.completions.create(
        model="gpt-4o-mini",  # Specify the model to use
        messages=[
            {
                "role": "user",
                "content": user_prompt,
            }  # User prompt is the input/query to respond to
        ],
        max_tokens=max_tokens,  # Max number of tokens to generate in the response
        temperature=temperature,  # Controls randomness in output
        top_p=top_p,  # Controls diversity via nucleus sampling
    )
    return completion.choices[
        0
    ].message.content  # Return the text content from the model's reply                                                        # Execute the function with the prompts

### Question 1: What is the protocol for managing sepsis in a critical care unit?


In [6]:
question_1 = "What is the protocol for managing sepsis in a critical care unit?"
base_prompt_response_1 = response(question_1)
base_prompt_response_1

"Managing sepsis in a critical care unit involves a systematic approach that is based on early identification, prompt treatment, and continuous monitoring. The following steps outline the general protocol for managing sepsis:\n\n### 1. Early Recognition\n- **Screening:** Utilize screening tools to identify patients at risk for sepsis, including those with infection signs, abnormal vital signs, or altered mental status.\n- **Assessment:** Evaluate the patient for symptoms of sepsis, such as fever, tachycardia, tachypnea, and hypotension.\n\n### 2. Initial Resuscitation\n- **Fluid Resuscitation:** Administer intravenous (IV) fluids (crystalloids) aggressively, typically 30 mL/kg within the first 3 hours for patients with septic shock.\n- **Vasopressors:** If hypotension persists despite adequate fluid resuscitation, initiate vasopressors (e.g., norepinephrine) to maintain mean arterial pressure (MAP) ≥ 65 mmHg.\n\n### 3. Source Control\n- **Identify Infection Source:** Conduct thorough i

In [7]:
# from IPython.display import Markdown, display
# display(Markdown(base_prompt_response_1))

### Question 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?


In [8]:
question_2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
base_prompt_response_2 = response(question_2)
base_prompt_response_2

'Common symptoms of appendicitis include:\n\n1. **Abdominal Pain**: Typically starts around the navel and then shifts to the lower right abdomen.\n2. **Loss of Appetite**: A sudden decrease in appetite is often reported.\n3. **Nausea and Vomiting**: These symptoms often follow the onset of abdominal pain.\n4. **Fever**: A mild fever may be present, often around 99°F to 102°F (37.2°C to 38.9°C).\n5. **Constipation or Diarrhea**: Some individuals may experience changes in bowel habits.\n6. **Inability to pass gas**: This may occur alongside abdominal pain.\n\nAppendicitis cannot be treated effectively with medication alone. While antibiotics can help manage infections, they do not resolve the underlying problem of the inflamed appendix. \n\nThe standard treatment for appendicitis is an **appendectomy**, which is the surgical removal of the appendix. There are two main types of appendectomy:\n\n1. **Open Appendectomy**: A larger incision is made in the lower right abdomen to remove the ap

### Question 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?


In [9]:
question_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
base_prompt_response_3 = response(question_3)
base_prompt_response_3

"Sudden patchy hair loss, often referred to as alopecia areata, can manifest as localized bald spots on the scalp or other areas of the body. Here are some effective treatments and solutions, as well as potential causes:\n\n### Possible Causes:\n1. **Autoimmune Disorders**: The immune system mistakenly attacks hair follicles.\n2. **Genetic Factors**: A family history of hair loss conditions can increase risk.\n3. **Stress**: Physical or emotional stress can trigger hair loss in some individuals.\n4. **Hormonal Changes**: Changes in hormones due to pregnancy, menopause, or thyroid issues.\n5. **Nutritional Deficiencies**: Lack of essential nutrients (e.g., iron, vitamin D, biotin).\n6. **Infections**: Fungal infections like tinea capitis can cause patchy hair loss.\n7. **Allergic Reactions**: Reactions to hair products or dyes can lead to hair loss.\n8. **Trichotillomania**: A psychological condition where individuals pull out their hair.\n\n### Effective Treatments:\n1. **Topical Treat

### Question 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?


In [10]:
question_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
base_prompt_response_4 = response(question_4)
base_prompt_response_4

'The treatment for a person who has sustained a physical injury to brain tissue, such as a traumatic brain injury (TBI), can vary significantly depending on the severity of the injury, the specific areas of the brain affected, and the resulting impairments. Here are some commonly recommended treatments and interventions:\n\n1. **Emergency Care**: \n   - **Stabilization**: Immediate treatment often focuses on stabilizing the individual, ensuring proper airway, breathing, and circulation.\n   - **Imaging**: CT or MRI scans may be performed to assess the extent of the injury and identify any bleeding or swelling.\n\n2. **Medical Management**:\n   - **Medications**: Anti-inflammatory drugs, anticonvulsants (to prevent seizures), and pain relievers may be prescribed.\n   - **Surgery**: In some cases, surgical intervention may be necessary to relieve pressure from swelling, remove blood clots, or repair damaged tissue.\n\n3. **Rehabilitation**:\n   - **Physical Therapy**: To improve motor sk

**Observations:**

- The responses do not tailor the advice to patient-specific variables like age, severity, or medical history, they stick to general treatment protocols or commonly known procedures.

- The answers provide basic overviews (e.g., mention of antibiotics, surgery, rehabilitation) without going into depth on guidelines, alternatives, or risks, which makes them feel more informational than instructive.


## Question Answering using LLM with Prompt Engineering


### Define a system prompt that aligns with the business problem


In [11]:
system_prompt = """
You are an AI assistant specializing in medical knowledge. Your role is to provide clear, precise, and medically reliable responses based on established medical guidelines and best practices.

When answering, prioritize factual correctness, align with widely accepted medical standards, and ensure clarity for both medical professionals and general users.
If a query requires specific reference materials beyond general medical knowledge, acknowledge the limitation rather than speculating.

"""

### Defining the function to Generate a Response From the LLM


In [12]:
# Define a function to get a response from the OpenAI chat model
def response(system_prompt, user_prompt, max_tokens=1000, temperature=0.75, top_p=0.95):
    # Create a chat completion using the OpenAI client
    completion = client.chat.completions.create(
        model="gpt-4o-mini",  # Specify the model to use (GPT-4o in this case)
        messages=[
            {
                "role": "system",
                "content": system_prompt,
            },  # System prompt sets the assistant's behavior
            {
                "role": "user",
                "content": user_prompt,
            },  # User prompt is the input/query to respond to
        ],
        max_tokens=max_tokens,  # Max number of tokens to generate in the response
        temperature=temperature,  # Controls randomness in output (0 = deterministic)
        top_p=top_p,  # Controls diversity via nucleus sampling
    )
    return completion.choices[
        0
    ].message.content  # Return the text content from the model's reply

### Question1: What is the protocol for managing sepsis in a critical care unit?


In [13]:
response_with_prompt_eng_1 = response(system_prompt, question_1)
response_with_prompt_eng_1

'The management of sepsis in a critical care unit follows established guidelines, particularly those from the Surviving Sepsis Campaign. The protocol can be summarized in several key steps:\n\n### 1. **Early Recognition and Diagnosis**\n   - **Screening**: Identify patients at risk using the quick Sequential Organ Failure Assessment (qSOFA) score or the Systemic Inflammatory Response Syndrome (SIRS) criteria.\n   - **Clinical Assessment**: Look for signs of infection, organ dysfunction, and septic shock.\n\n### 2. **Initial Resuscitation**\n   - **Fluid Resuscitation**: Administer intravenous fluids (crystalloids) aggressively in the first hour, typically 30 mL/kg for adults, to restore hemodynamic stability.\n   - **Vasopressors**: If hypotension persists despite adequate fluid resuscitation, initiate vasopressors (e.g., norepinephrine) to maintain a mean arterial pressure (MAP) of ≥65 mmHg.\n\n### 3. **Source Control**\n   - **Identify and Treat Underlying Infection**: Obtain culture

### Question2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?


In [14]:
response_with_prompt_eng_2 = response(system_prompt, question_2)
response_with_prompt_eng_2

"Common symptoms of appendicitis include:\n\n1. **Abdominal Pain**: Typically starts around the navel and then moves to the lower right abdomen (McBurney's point).\n2. **Nausea and Vomiting**: Often follows the onset of abdominal pain.\n3. **Loss of Appetite**: Patients may not feel like eating.\n4. **Fever**: Low-grade fever may develop.\n5. **Constipation or Diarrhea**: Changes in bowel habits can occur.\n6. **Abdominal Swelling**: In some cases, there may be swelling or tenderness in the abdomen.\n\nAppendicitis cannot be effectively treated with medication alone. The definitive treatment is usually surgical removal of the appendix, a procedure known as an **appendectomy**. This can be performed either through an open surgery method or laparoscopically (minimally invasive surgery).\n\nIf appendicitis is suspected, it's essential to seek medical attention promptly, as a delayed diagnosis may lead to complications such as a ruptured appendix, which can result in peritonitis and sepsis

### Question3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?


In [15]:
response_with_prompt_eng_3 = response(system_prompt, question_3)
response_with_prompt_eng_3

"Sudden patchy hair loss, often characterized by localized bald spots on the scalp, is commonly associated with a condition called alopecia areata. Here’s an overview of potential causes, treatments, and solutions:\n\n### Possible Causes:\n1. **Alopecia Areata**: An autoimmune condition where the immune system attacks hair follicles, leading to sudden hair loss.\n2. **Stress**: Physical or emotional stress can trigger hair loss.\n3. **Genetics**: A family history of hair loss may predispose individuals to alopecia areata.\n4. **Thyroid Disorders**: Conditions such as hypothyroidism or hyperthyroidism can contribute to hair loss.\n5. **Nutritional Deficiencies**: Lack of essential nutrients, particularly iron, vitamin D, and certain B vitamins, can affect hair health.\n6. **Infections**: Fungal infections like tinea capitis can cause patchy hair loss.\n7. **Other Autoimmune Conditions**: Diseases like lupus or vitiligo may be associated with hair loss.\n\n### Effective Treatments:\n1. *

### Question4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?


In [16]:
response_with_prompt_eng_4 = response(system_prompt, question_4)
response_with_prompt_eng_4

'The treatment for a person who has sustained a physical injury to brain tissue, such as a traumatic brain injury (TBI), depends on the severity of the injury and the specific symptoms or impairments that result. Here are some commonly recommended approaches:\n\n1. **Immediate Medical Care:**\n   - **Stabilization:** Ensuring the patient’s airway, breathing, and circulation are stable.\n   - **Imaging Studies:** CT scans or MRIs may be performed to assess the extent of the injury.\n\n2. **Surgical Interventions:**\n   - If there is significant swelling, bleeding, or pressure on the brain (e.g., hematoma), surgery may be necessary to relieve pressure and repair damaged areas.\n\n3. **Medication:**\n   - **Pain Management:** Analgesics may be prescribed to manage pain.\n   - **Anti-Inflammatory Drugs:** Medications like corticosteroids may be used to reduce swelling in the brain.\n   - **Anticonvulsants:** If seizures are a risk, these medications may be prescribed.\n\n4. **Rehabilitatio

**Observations:**

**Question1: Sepsis Management in Critical Care**

- **Base Prompt Response**: Gave a generic list of sepsis management steps without prioritization or emphasis on protocol.
- **Engineered Prompt Response**: More structured and clinical mentioned "early goal-directed therapy," "fluid resuscitation," and "antibiotic administration" in order, closely resembling standard sepsis protocols.

- Improved in clinical depth and sequencing.

**Question2: Appendicitis Symptoms and Treatment**

- **Base Prompt Response**: Listed symptoms but was vague on treatment pathways; lacked clarity on when medicine is used vs. surgery.
- **Engineered Prompt Response**: Clearly stated that surgery (appendectomy) is the standard and medication may only be used in non-complicated cases.

- Improved in decision-making clarity and completeness.

**Question3: Patchy Hair Loss (Alopecia Areata)**

- **Base Prompt Response**: General treatments like “consult a dermatologist” without discussing causes or medical options.
- **Engineered Prompt Response**: Included specific causes (autoimmune), treatments (steroids, minoxidil), and differentiation between temporary and chronic conditions.

- Improved in specificity and cause-treatment mapping.

**Question4: Brain Injury Treatment**

- **Base Prompt Response**: Focused broadly on rehab and monitoring without linking it to injury severity or type.
- **Engineered Prompt Response**: Mentioned both acute interventions (e.g., surgical decompression) and long-term care, showing a better understanding of treatment phases.
- Improved in handling both acute and chronic dimensions of treatment


## Data Preparation for RAG


### Loading the Data


In [17]:
# uncomment and run the below code snippets if the dataset is present in the Google Drive
# from google.colab import drive
# drive.mount('/content/drive')

In [18]:
# Set the path to the PDF file
manual_pdf_path = (
    "medical_diagnosis_manual.pdf"  # Path to the medical diagnosis manual PDF
)

# Load the PDF using LangChain's PyPDFLoader
pdf_loader = PyMuPDFLoader(
    manual_pdf_path
)  # Initialize the PDF loader with the file path

# Extract content from the PDF
manual = pdf_loader.load()  # Load and extract text from all pages of the PDF

### Data Overview


#### Checking the first 5 pages


In [19]:
# Loop through the first 5 pages of the PDF content
for i in range(5):
    print(f"Page Number : {i+1}", end="\n")  # Print the page number (1-based index)
    print(
        manual[i].page_content, end="\n"
    )  # Print the content of the corresponding page

Page Number : 1
ali@zarreh.ai
MR-ALIREZAZ
This file is meant for personal use by ali@zarreh.ai only.
Sharing or publishing the contents in part or full is liable for legal action.
Page Number : 2
ali@zarreh.ai
MR-ALIREZAZ
This file is meant for personal use by ali@zarreh.ai only.
Sharing or publishing the contents in part or full is liable for legal action.
Page Number : 3
Table of Contents
1
Front    ................................................................................................................................................................................................................
1
Cover    .......................................................................................................................................................................................................
2
Front Matter    .............................................................................................................................................................

### Data Chunking


#### Chunk the PDF into Manageable Text Sections Using a Token-Based Splitter


In [20]:
# Initialize a text splitter that uses OpenAI's token encoder
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base",  # Encoding used by popular LLMs
    chunk_size=256,  # Each chunk will have up to 256 tokens
    chunk_overlap=20,  # 20 tokens will overlap between consecutive chunks (for context continuity)
)

#### Split the Loaded PDF into Chunks for Further Processing


In [21]:
# Use the text splitter to divide the PDF content into smaller chunks
document_chunks = pdf_loader.load_and_split(
    text_splitter
)  # Returns a list of chunked documents

#### Check the Number of Chunks Created


In [22]:
len(document_chunks)  # Total number of text chunks generated from the PDF

15558

### Generate Vector Embeddings for Text Chunks Using OpenAI


In [23]:
# Initialize the OpenAI Embeddings model with API credentials
embedding_model = OpenAIEmbeddings(
    openai_api_key=API_KEY,  # Your OpenAI API key for authentication
    openai_api_base=OPENAI_API_BASE,  # The OpenAI API base URL endpoint
)

# Generate embeddings (vector representations) for the first two document chunks
embedding_1 = embedding_model.embed_query(
    document_chunks[0].page_content
)  # Embedding for chunk 0
embedding_2 = embedding_model.embed_query(
    document_chunks[1].page_content
)  # Embedding for chunk 1

# Check and print the dimension (length) of the embedding vector
print(
    "Dimension of the embedding vector ", len(embedding_1)
)  # Typically 1536 or 2048 depending on model

/tmp/ipykernel_26943/1548090435.py:2: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import OpenAIEmbeddings``.
  embedding_model = OpenAIEmbeddings(


Dimension of the embedding vector  1536


In [24]:
# Verify if both embeddings have the same dimension (should be True)
len(embedding_1) == len(embedding_2)

# Return/display the two embedding vectors for further inspection or use
embedding_1, embedding_2

([-0.012080292646606447,
  -0.0019911977704216203,
  -0.01654834545334951,
  -0.023872787980980727,
  0.0046227445605598085,
  0.012785396061575251,
  -0.012526378480566303,
  -0.02027532157807866,
  -0.004867372275957149,
  -0.025944928629052316,
  0.02092286646192364,
  0.005730764212653645,
  0.016087869753778045,
  0.0035327122404804822,
  0.007504315382115016,
  -0.005608450354954975,
  0.0015424137784519243,
  -0.018059282273890986,
  0.009339022781933767,
  0.014555349531803069,
  0.00963401549263304,
  0.010346314306068954,
  -0.01830390905796572,
  -0.009187929658673185,
  -0.010734840677582376,
  -0.01462010439271661,
  0.019771675350349762,
  -0.035399069404556345,
  0.03246353681978826,
  -0.04616269260732976,
  0.03675171863469274,
  0.0013805277903213313,
  -0.0020577507824599823,
  -0.018908283413653266,
  -0.023800839584245292,
  0.015814463238480094,
  0.04279546219156821,
  0.0025290188812401534,
  0.025642742382531152,
  -0.013562448804602098,
  0.009461337105293741,

### Vector Database Creation


#### Setup Vector Store Directory


In [25]:
# Creating a folder for saving the vector DB so it persists between runs
out_dir = "medical_db"  # Directory to store the persistent vector database

# Create the directory if it doesn't exist
if not os.path.exists(out_dir):
    os.makedirs(out_dir)  # Make directory to save vector store files

#### Create Vector Store from Documents


In [26]:
# Building the vector store and saving it to disk for future use
vectorstore = Chroma.from_documents(
    document_chunks,  # Documents to index
    embedding_model,  # Embedding model for converting text to vectors
    persist_directory=out_dir,  # Save vector DB files here
)

#### Load Vector Store


In [27]:
# Reloading the vector store from disk without recomputing embeddings
vectorstore = Chroma(
    persist_directory=out_dir,  # Load existing vector DB files
    embedding_function=embedding_model,  # Use the same embedding function for queries
)

/tmp/ipykernel_26943/2975314053.py:2: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorstore = Chroma(


#### Explore Vector Store and Perform Searches


In [28]:
# Inspect the embedding function in use
vectorstore.embeddings

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x7179c5e79df0>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x7179c57170b0>, model='text-embedding-ada-002', deployment='text-embedding-ada-002', openai_api_version='', openai_api_base='https://aibe.mygreatlearning.com/openai/v1', openai_api_type='', openai_proxy='', embedding_ctx_length=8191, openai_api_key='gl-U2FsdGVkX1+txoaBWx1SLxwKCkKQt5lxQ7ao2XnljPgFSWo74L9SPwlDirRb1sW8', openai_organization=None, allowed_special=set(), disallowed_special='all', chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None)

In [29]:
# Search for top 3 most relevant chunk for the query
vectorstore.similarity_search(
    "What are the common symptoms and treatments for pulmonary embolism?", k=3
)

[Document(metadata={'keywords': '', 'format': 'PDF 1.7', 'modDate': 'D:20260523224102Z', 'creator': 'Atop CHM to PDF Converter', 'creationDate': 'D:20120615054440Z', 'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition', 'moddate': '2026-05-23T22:41:02+00:00', 'source': 'medical_diagnosis_manual.pdf', 'file_path': 'medical_diagnosis_manual.pdf', 'trapped': '', 'creationdate': '2012-06-15T05:44:40+00:00', 'total_pages': 4114, 'page': 2079, 'author': '', 'subject': ''}, page_content='Chapter 194. Pulmonary Embolism\nIntroduction\nPulmonary embolism (PE) is the occlusion of ≥ 1 pulmonary arteries by thrombi that originate\nelsewhere, typically in the large veins of the lower extremities or pelvis. Risk factors are\nconditions that impair venous return, conditions that cause endothelial injury or dysfunction,\nand underlying hypercoagulable states. Symptoms are nonspecific and include dyspnea,\npleuritic chest pain, cou

### Retrieval and Response Generation using Vector Search


#### Convert Vector Store into a Retriever and Retrieve Relevant Documents


In [30]:
# Wraping the vector store into a retriever object to fetch the most relevant documents for a given query using similarity search
retriever = vectorstore.as_retriever(
    search_type="similarity",  # Use similarity search (based on vector distance)
    search_kwargs={"k": 3},  # Retrieve top 2 most relevant documents
)

#### System and User Prompt Template


Prompts guide the model to generate accurate responses. Here, we define two parts:

    1. The system message describing the assistant's role.
    2. A user message template including context and the question.


In [31]:
# Define the system prompt for the model
qna_system_message = """
You are an AI assistant designed to support professional doctors at St. Bernard's Medical Center. Your task is to provide evidence-based, concise, and relevant medical information to doctors' clinical questions based on the context provided.

User input will include the necessary context for you to answer their questions. This context will begin with the token: ###Context. The context contains references to specific portions of trusted medical literature and research articles relevant to the query, along with their source details.

When crafting your response:
1. Use only the provided context to answer the question.
2. If the answer is found in the context, respond with concise and actionable medical insights.
3. Include the source reference with the page number, journal name, or publication, as provided in the context.
4. If the question is unrelated to the context or the context is empty, clearly respond with: "Sorry, this is out of my knowledge base."

Please adhere to the following response guidelines:
- Provide clear, direct answers using only the given context.
- Do not include any additional information outside of the context.
- Avoid rephrasing or summarizing the context unless explicitly relevant to the question.
- If no relevant answer exists in the context, respond with: "Sorry, this is out of my knowledge base."
- If the context is not provided, your response should also be: "Sorry, this is out of my knowledge base."

Here is an example of how to structure your response:

Answer:
[Medical answer based on context]

Source:
[Source details with page or section]
"""

In [32]:
# Define the user message template
qna_user_message_template = """
###Context
Here are some excerpts from medical literature and their sources that are relevant to the clinical question mentioned below:
{context}

###Question
{question}
"""

### Response Function


In [33]:
def generate_rag_response(
    user_input, k=3, max_tokens=1000, temperature=0.75, top_p=0.95
):
    global qna_system_message, qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input, k=k)
    context_list = [d.page_content for d in relevant_document_chunks]

    # Combine document chunks into a single context
    context_for_query = ". ".join(context_list)

    user_message = qna_user_message_template.replace("{context}", context_for_query)
    user_message = user_message.replace("{question}", user_input)

    # Generate the response
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": qna_system_message},
                {"role": "user", "content": user_message},
            ],
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
        )
        # Extract and print the generated text from the response
        response = response.choices[0].message.content.strip()
    except Exception as e:
        response = f"Sorry, I encountered the following error: \n {e}"

    return response

## Question Answering using RAG


### Question1: What is the protocol for managing sepsis in a critical care unit?


In [34]:
response_with_rag_1 = generate_rag_response(question_1)
response_with_rag_1

/tmp/ipykernel_26943/3455360944.py:6: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  relevant_document_chunks = retriever.get_relevant_documents(query=user_input, k=k)


'Answer:\nThe protocol for managing sepsis in a critical care unit includes initiating very prompt empiric therapy immediately after suspecting sepsis, which is essential and may be lifesaving. Parenteral antibiotics should be administered after obtaining specimens for Gram stain and culture. Antibiotic selection should be based on an educated guess considering the suspected source, clinical setting, knowledge of causative organisms, sensitivity patterns common to the unit, and previous culture results. A recommended regimen for septic shock of unknown cause includes gentamicin or tobramycin (5.1 mg/kg IV once/day) plus a 3rd-generation cephalosporin (cefotaxime 2 g every 6 to 8 hours or ceftriaxone 2 g once/day). If Pseudomonas is suspected, ceftazidime (2 g IV every 8 hours) should be used. Alternatively, ceftazidime can be combined with a fluoroquinolone (e.g., ciprofloxacin). Monotherapy options include maximal therapeutic doses of ceftazidime (2 g IV every 8 hours) or imipenem (1 

In [36]:
from IPython.display import HTML, display

html = f"""
<div style="display:grid; grid-template-columns: repeat(3, 1fr); gap: 16px;">
  <div style="padding: 16px; border: 1px solid #ddd; border-radius: 8px; background: #f9f9f9;">
    <h3 style="margin-top:0;">Base Prompt</h3>
    <pre style="white-space: pre-wrap; word-break: break-word;">{base_prompt_response_1}</pre>
  </div>
  <div style="padding: 16px; border: 1px solid #ddd; border-radius: 8px; background: #f9f9f9;">
    <h3 style="margin-top:0;">Prompt Engineered</h3>
    <pre style="white-space: pre-wrap; word-break: break-word;">{response_with_prompt_eng_1}</pre>
  </div>
  <div style="padding: 16px; border: 1px solid #ddd; border-radius: 8px; background: #f9f9f9;">
    <h3 style="margin-top:0;">RAG</h3>
    <pre style="white-space: pre-wrap; word-break: break-word;">{response_with_rag_1}</pre>
  </div>
</div>
"""
display(HTML(html))


### Question2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?


In [ ]:
response_with_rag_2 = generate_rag_response(question_2)
response_with_rag_2

In [ ]:
from IPython.display import HTML, display

html = f"""
<div style="display:grid; grid-template-columns: repeat(3, 1fr); gap: 16px;">
  <div style="padding: 16px; border: 1px solid #ddd; border-radius: 8px; background: #f9f9f9;">
    <h3 style="margin-top:0;">Base Prompt</h3>
    <pre style="white-space: pre-wrap; word-break: break-word;">{base_prompt_response_2}</pre>
  </div>
  <div style="padding: 16px; border: 1px solid #ddd; border-radius: 8px; background: #f9f9f9;">
    <h3 style="margin-top:0;">Prompt Engineered</h3>
    <pre style="white-space: pre-wrap; word-break: break-word;">{response_with_prompt_eng_2}</pre>
  </div>
  <div style="padding: 16px; border: 1px solid #ddd; border-radius: 8px; background: #f9f9f9;">
    <h3 style="margin-top:0;">RAG</h3>
    <pre style="white-space: pre-wrap; word-break: break-word;">{response_with_rag_2}</pre>
  </div>
</div>
"""
display(HTML(html))


### Question3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?


In [ ]:
response_with_rag_3 = generate_rag_response(question_3)
response_with_rag_3

In [ ]:
from IPython.display import HTML, display

html = f"""
<div style="display:grid; grid-template-columns: repeat(3, 1fr); gap: 16px;">
  <div style="padding: 16px; border: 1px solid #ddd; border-radius: 8px; background: #f9f9f9;">
    <h3 style="margin-top:0;">Base Prompt</h3>
    <pre style="white-space: pre-wrap; word-break: break-word;">{base_prompt_response_3}</pre>
  </div>
  <div style="padding: 16px; border: 1px solid #ddd; border-radius: 8px; background: #f9f9f9;">
    <h3 style="margin-top:0;">Prompt Engineered</h3>
    <pre style="white-space: pre-wrap; word-break: break-word;">{response_with_prompt_eng_3}</pre>
  </div>
  <div style="padding: 16px; border: 1px solid #ddd; border-radius: 8px; background: #f9f9f9;">
    <h3 style="margin-top:0;">RAG</h3>
    <pre style="white-space: pre-wrap; word-break: break-word;">{response_with_rag_3}</pre>
  </div>
</div>
"""
display(HTML(html))


### Question4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?


In [ ]:
response_with_rag_4 = generate_rag_response(question_4)
response_with_rag_4

In [ ]:
from IPython.display import HTML, display

html = f"""
<div style="display:grid; grid-template-columns: repeat(3, 1fr); gap: 16px;">
  <div style="padding: 16px; border: 1px solid #ddd; border-radius: 8px; background: #f9f9f9;">
    <h3 style="margin-top:0;">Base Prompt</h3>
    <pre style="white-space: pre-wrap; word-break: break-word;">{base_prompt_response_4}</pre>
  </div>
  <div style="padding: 16px; border: 1px solid #ddd; border-radius: 8px; background: #f9f9f9;">
    <h3 style="margin-top:0;">Prompt Engineered</h3>
    <pre style="white-space: pre-wrap; word-break: break-word;">{response_with_prompt_eng_4}</pre>
  </div>
  <div style="padding: 16px; border: 1px solid #ddd; border-radius: 8px; background: #f9f9f9;">
    <h3 style="margin-top:0;">RAG</h3>
    <pre style="white-space: pre-wrap; word-break: break-word;">{response_with_rag_4}</pre>
  </div>
</div>
"""
display(HTML(html))


**Question 1 - Sepsis Management**

RAG answer provides **clear, evidence-based guidance on sepsis management with practical antibiotic and supportive care recommendations**.

**Question 2 - Appendicitis Symptoms and Treatment**

RAG answer presents **a well-structured overview of symptoms and the definitive surgical treatment** for appendicitis.

**Question 3 - Patchy Hair Loss (Alopecia Areata)**

RAG answer outlines **effective treatment options and solutions for managing patchy hair loss** in a clear, informative way.

**Question 4 - Brain Injury Treatment**

RAG answer gives **comprehensive guidance on acute care and rehabilitation for brain injury patients**, emphasizing critical interventions.


## Actionable Insights and Business Recommendations


1. **Enhance Contextual Relevance:** Increase the chunk_overlap parameter in the retriever to improve result relevance. Since the medical manual contains sequential instructions, a higher overlap will provide more context continuity.

2. **Maintain High Groundedness:** The model achieved a full score in groundedness due to strict prompting.

3. **Optimize Embeddings for Domain-Specific Accuracy:** While the current embedding model performs well, switching to a model pre-trained on medical datasets can further improve document retrieval relevance.

4. **Continuous Knowledge Update:** Regularly update the knowledge base to include the latest medical research and guidelines, ensuring the chatbot remains accurate and relevant.

5. **Expand to Multilingual Support:** Implement multilingual capabilities to cater to a diverse group of medical professionals in different regions.

6. **Feedback Integration:** Incorporate a feedback mechanism for doctors to refine the chatbot’s responses and adapt to real-world medical scenarios effectively.

7. **Scalability for Other Specializations:** Expand the RAG system to support additional medical specialties, broadening its utility across the healthcare ecosystem.


<font size=6 color='blue'>Power Ahead</font>

---
